# 65 — F1 & Confusion Matrix
**Goal:** Multi-class evaluation with confusion matrices.

Chapter 64 measured extraction against a single set of gold entities. Real resume pipelines also *classify* — assigning each bullet or section a label (skill, experience, education, summary). This chapter generalizes precision/recall/F1 to the multi-class setting and introduces the **confusion matrix**, the canonical grid that shows not just *how many* errors occur, but *which pairs of classes* get confused.

**Why it matters for resumes / ATS:** section and entity classification is the backbone of structured profile building — an ATS must know which text is a skill and which is a work experience before it can match anything. The confusion matrix reveals the *direction* of errors: "experience classified as skill" and "skill classified as experience" are the classic swaps, and a matrix exposes them at a glance. When classes are imbalanced — most resumes contain far more skill mentions than summary blocks — the averaging scheme you pick for F1 changes what your headline number actually means.

## 1. Confusion Matrix Basics

A **confusion matrix** is a grid where rows are true labels, columns are predicted labels, and the diagonal holds the correct classifications. Every off-diagonal cell names a specific error pair — read a row to see what a class *was mistaken for*, read a column to see what *got mislabeled as* it. For binary problems it degenerates to the four counts tp/fp/fn/tn; for multi-class problems it is the only compact way to see the full error structure at once.

**What the code does:** it builds gold and predicted label lists for a 10-bullet resume (4 skill, 3 experience, 2 education, 1 summary) and calls `confusion_matrix(..., labels=classes)` with an explicit label order, then prints the grid by hand without matplotlib. Running it gives `[[3,1,0,0],[1,2,0,0],[0,0,2,0],[0,0,0,1]]`: 8 of 10 bullets on the diagonal (80% accuracy), and both errors are skill/experience swaps — one true-skill bullet labeled experience, one true-experience bullet labeled skill.

**Try it:** the `labels=` argument matters — without it sklearn infers an arbitrary order from the data, and a matrix whose axes are sorted differently is unreadable.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Gold labels and predictions for resume section classification
true_labels = ["skill", "experience", "education", "skill", "experience", "summary", "skill", "education", "experience", "skill"]
pred_labels = ["skill", "experience", "education", "skill", "skill", "summary", "experience", "education", "experience", "skill"]

classes = ["skill", "experience", "education", "summary"]
cm = confusion_matrix(true_labels, pred_labels, labels=classes)
print("Confusion Matrix:")
print(f"         {'':8s}{''.join(f'{c:10s}' for c in classes)}")
for i, row in enumerate(cm):
    print(f"  {classes[i]:10s}{''.join(f'{val:10d}' for val in row)}")

## 2. Classification Report

The matrix shows *where* errors land; the **classification report** turns each row into the per-class precision, recall, and F1 from Chapter 64, plus `support` — the number of true instances of each class. Per-class numbers are the honest view: the overall accuracy of 0.80 is dragged down entirely by the two skill/experience confusions, and the per-class F1 tells you which class suffers most and from what.

**What the code does:** one `classification_report()` call over the same labels. The run reports: skill P/R/F1 = 0.75 on support 4 (three gold bullets kept, one lost to the swap), experience 0.67 on support 3 (two of three), education 1.00 on support 2, summary 1.00 on support 1. The macro average (0.85) exceeds the weighted average (0.80) precisely because the small, easy classes — education and summary — get equal votes under macro but barely move weighted.

**Try it:** compare the report to the matrix cell-by-cell — each off-diagonal cell must show up as a precision loss in its column's class and a recall loss in its row's class.

In [ ]:
report = classification_report(true_labels, pred_labels, labels=classes)
print("Classification Report:")
print(report)

## 3. Macro vs Micro vs Weighted F1

With several classes you must decide how to combine their F1 scores, and the choice is not cosmetic:

| Average | How it is computed | Bias |
|---|---|---|
| `macro` | mean of per-class F1, each class equal weight | rare classes count as much as common ones |
| `micro` | global tp/fp/fn pooled across all classes | dominated by the most common class |
| `weighted` | per-class F1 weighted by support | reflects the real data distribution |

**What the code does:** `f1_score(y_true, y_pred, average=...)` on a 10-sample numeric dataset returns macro 0.802, micro 0.800, weighted 0.797. They differ only in the third decimal because the classes are near-balanced; on a real resume corpus, where "skill" massively outnumbers "summary", the gap widens. The notebook's guidance is the right default: **use weighted F1** so the headline reflects the distribution you actually serve.

**Try it:** flip two labels between the minority and majority classes and watch macro move far more than micro — that is the "rare class gets a vote" effect.

In [ ]:
from sklearn.metrics import f1_score

y_true = [0, 0, 1, 1, 2, 2, 0, 1, 2, 0]
y_pred = [0, 1, 1, 1, 2, 2, 0, 1, 0, 0]

print("F1 Scoring methods:")
print(f"  Macro-F1:    {f1_score(y_true, y_pred, average='macro'):.3f} (each class equal weight)")
print(f"  Micro-F1:    {f1_score(y_true, y_pred, average='micro'):.3f} (global TP/FP/FN)")
print(f"  Weighted-F1: {f1_score(y_true, y_pred, average='weighted'):.3f} (weighted by support)")
print()
print("For resume tasks, use Weighted-F1 when classes are imbalanced.")
print("Most resumes have more 'skill' entities than 'summary'.")

## Summary: Confusion matrices show WHERE errors happen. Use weighted F1 for imbalanced resume data.

**The matrix names your errors; the averaging scheme decides what your score means.**

Two bullets mislabeled in a skill/experience swap is not noise — it is a systematic signature of a classifier that cannot reliably separate adjacent section types, and the matrix surfaces it in two cells instead of burying it in an accuracy number. Per-class reports localize the damage, and weighted F1 keeps the headline honest on skewed corpora. The next chapter shifts from measuring *wrong answers* to measuring *made-up answers*: hallucination testing, where the failure is not classification error but output that is not grounded in the resume at all.